This example considers a toy problem generated by ChatGPT 5.1 to come up with a problem that may cause scaling issues with parameter size. The model considered is a one state, two input (one fixed, other doe design variable), six parameter system:

$\dot{x}(t) = a_1 x(t) + a_2 u_1(t) + a_3 u_1 ^2(t) + a_4 u_2(t) + a_5 u_2 ^2 (t) + a_6 e^{-x(t)}$

Real values of the parameters:

PARAMETERS:
1. $a_1 = -1 \in [-2,-0.1]$
2. $a_2 = 1.5 \in [0,3]$
3. $a_3 = -0.5 \in [-2,2]$
4. $a_4 = 2 \in [0,3]$
5. $a_5 = -1 \in [-2,2]$
6. $a_6 = 0.8 \in [0,2]$

STATE AND CONTROL INPUT:
1. $x \in [0,5]$
2. $u_1 \in [0,1]$
3. $u_2 \in [0,1]$


This model solves for the unscaled version but with bounds. 

In [1]:
"Modifications to avoid the IPOPT Error on CRC"

import shutil
import pyomo.environ as pyo

IPOPT_BIN = shutil.which("ipopt")

def make_ipopt():
    set = pyo.SolverFactory("ipopt", executable = IPOPT_BIN)
    set.options["linear_solver"] = "ma57"
    return set

In [2]:

from pyomo.dae import ContinuousSet, DerivativeVar, Simulator
import numpy as np
from pathlib import Path
from pyomo.contrib.parmest.experiment import Experiment



# ========================
class SixParameterExperiment(Experiment):
    def __init__(self, data, nfe, ncp):
        self.data = data
        self.nfe = nfe
        self.ncp = ncp
        self.model = None


    def get_labeled_model(self):
        if self.model is None:
            self.create_model()
            self.finalize_model()
            self.label_experiment()
        return self.model

   
    def create_model(self):
     

        m = self.model = pyo.ConcreteModel()

        # Model parameters

       
        m.t = ContinuousSet(bounds=[0, 1])

        # State and input
        m.x = pyo.Var(m.t, within=pyo.Reals, bounds = [0,5])
        m.u1 = pyo.Var(m.t, within=pyo.Reals, bounds = [0,1])
        m.u2 = pyo.Var(m.t, within=pyo.Reals, bounds = [0,1])

        # Unknown parameter
        m.a1 = pyo.Var(within=pyo.Reals, bounds = [-2,-0.1])
        m.a2 = pyo.Var(within=pyo.Reals, bounds = [0,3])
        m.a3 = pyo.Var(within=pyo.Reals, bounds = [-2,2])
        m.a4 = pyo.Var(within=pyo.Reals, bounds = [0,3])
        m.a5 = pyo.Var(within=pyo.Reals, bounds = [-2,2])
        m.a6 = pyo.Var(within=pyo.Reals, bounds = [0,2])

        # Differential variables wrt x
        m.dxdt = DerivativeVar(m.x, wrt=m.t)
       

    

        # State odes
        @m.Constraint(m.t)
        def x_ode(m, t):
            return m.dxdt[t] == -m.a1 * m.x[t] + m.a2*m.u1[t] + m.a3*m.u1[t]**2  + m.a4*m.u2[t] + m.a5*m.u2[t]**2 + m.a6*pyo.exp(-m.x[t])
            


    def finalize_model(self):
      
        m = self.model

      
        control_points1 = self.data["control_points1"]
        control_points2 = self.data["control_points2"]

      
        m.x[0].value = self.data["x0"] # Initial state
        m.x[0].fix()

        # Update model time `t` with time range and control time points
        m.t.update(self.data["t_range"])
        m.t.update(control_points1)

        # Fix the unknown parameter values
        m.a1.fix(self.data["a1"])
        m.a2.fix(self.data["a2"])
        m.a3.fix(self.data["a3"])
        m.a4.fix(self.data["a4"])
        m.a5.fix(self.data["a5"])
        m.a6.fix(self.data["a6"])
        # m.u2.fix()
        

        m.t_control = control_points1

        # Discretizing the model
        discr = pyo.TransformationFactory("dae.collocation")
        discr.apply_to(m, nfe=self.nfe, ncp=self.ncp, wrt=m.t)

        # Initializing control input in the model
        cv1 = None
        for t in m.t:
            if t in control_points1:
                cv1 = control_points1[t]
                m.u1[t].fix()
            # m.u1[t].setlb(self.data["u_bounds"][0])
            # m.u1[t].setub(self.data["u_bounds"][1])
            m.u1[t] = cv1
        cv2 = None
        for t in m.t:
            if t in control_points2:
                cv2 = control_points2[t]
                m.u2[t].fix()
            # m.u1[t].setlb(self.data["u_bounds"][0])
            # m.u1[t].setub(self.data["u_bounds"][1])
            m.u2[t] = cv2
        

        # Make a constraint that holds control input constant between control time points
        @m.Constraint(m.t - control_points1)
        def u1_control(m, t):
            """
            Piecewise constant control input between control points
            """
            neighbour_t = max(tc for tc in control_points1 if tc < t)
            return m.u1[t] == m.u1[neighbour_t]
        @m.Constraint(m.t - control_points2)
        def u2_control(m, t):
            """
            Piecewise constant control input between control points
            """
            neighbour_t = max(tc for tc in control_points2 if tc < t)
            return m.u2[t] == m.u2[neighbour_t]


        

        #########################
        # End model finalization

    def label_experiment(self):
        """
        Example for annotating (labeling) the model with a
        full experiment.
        """
        m = self.model

        # Set measurement labels
        m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        
        m.experiment_outputs.update((m.x[t], None) for t in m.t_control)


        # Adding error for measurement values (assuming no covariance and constant error for all measurements)
        m.measurement_error = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        meas_error = 1e-2  # Error in state measurement

        m.measurement_error.update((m.x[t], meas_error) for t in m.t_control)
   

        # Identify design variables (experiment inputs) for the model
        m.experiment_inputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
    
        # Add experimental input label for control input
        m.experiment_inputs.update((m.u1[t], None) for t in m.t_control)
        m.experiment_inputs.update((m.u2[t], None) for t in m.t_control)


        # Add unknown parameter labels
        m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        # Add labels to all unknown parameters with nominal value as the value
        m.unknown_parameters.update((k, pyo.value(k)) for k in [m.a1, m.a2, m.a3, m.a4, m.a5, m.a6])

        #########################
        # End model labeling

In [ ]:


from pyomo.common.dependencies import numpy as np

from pyomo.contrib.doe import DesignOfExperiments

import pyomo.environ as pyo
from pyomo.environ import (
    ConcreteModel,
    Var,
    Param,
    Constraint,
    TransformationFactory,
    SolverFactory,
    Objective,
    minimize,
    value as pyovalue,
    Suffix,
    Expression,
    sin,
    NonNegativeReals,
)



data_ex = {"x0": 1, "x_bounds": [0, 5], "t_range": [0, 1],
           "control_points1": {"0": 0.5, "0.125": 0.5, "0.25": 0.5, "0.375": 0.5, "0.5": 0.5, "0.625": 0.5, "0.75": 0.5,
                              "0.875": 0.5, "1": 0.5}, "u_bounds": [0, 1],
           "control_points2": {"0": 0.5, "0.125": 0.5, "0.25": 0.5, "0.375": 0.5, "0.5":0.5, "0.625": 0.5, "0.75": 0.5,
                              "0.875": 0.5, "1": 0.5},
           "a1": -0.8, "a2":1.2, "a3": -0.3, "a4": 1.8, "a5": -0.7, "a6": 0.6}
# Put control input control time points into correct format for two parameter experiment
data_ex["control_points1"] = {
    float(k): v for k, v in data_ex["control_points1"].items()
}

data_ex["control_points2"] = {
    float(k): v for k, v in data_ex["control_points2"].items()
}

# Create a two parameterExperiment object; data and discretization information are part
# of the constructor of this object
experiment = SixParameterExperiment(data=data_ex, nfe=10, ncp=3)

# Use a central difference, with step size 1e-3
fd_formula = "central"
step_size = 1e-3


objective_option = "determinant"
scale_nominal_param_value = True



"Calling the Doe object 1000 times, saving to an excel file"
## Importing required packages

from pathlib import Path
from openpyxl import Workbook, load_workbook

NOTEBOOK_ID = "Sixpmt" # Defining the notebook ID for excel sheet tagging
SCENARIO = "Symbolic" # Scenario implies the environment


"Creating a path object and creating an excel file to save the output of the runs"
RESULTS_DIR = Path("results") ## Foldername
RESULTS_DIR.mkdir(exist_ok=True)
XLSX_PATH = Path("results")/ f"{NOTEBOOK_ID}_{SCENARIO}.xlsx" ## Create an excel sheet with the 
# notebook ID and scenario

SHEET_NAME = "Data" ## Name of the sheet within the excel file


## Creating the excel file


wb = Workbook()
ws = wb.active
ws.title = SHEET_NAME

design_names = None # specifies the column names, will be filled at the first run



N_Runs = 1000 ## Number of runs in this code

for run in range(1, N_Runs+1):
    print(f"Run {run}/{N_Runs}")
    doe_obj = DesignOfExperiments(
    experiment,
    fd_formula=fd_formula,
    step=step_size,
    objective_option=objective_option,
    scale_constant_value=1,
    scale_nominal_param_value=scale_nominal_param_value,
    gradient_method = "pynumero",
    prior_FIM=None,
    jac_initial=None,
    fim_initial=None,
    L_diagonal_lower_bound=1e-7,
    solver= make_ipopt(),#SolverFactory('IPOPT', options={'linear_solver': 'mumps',}), #'print_options_documentation' : 'yes', 'print_level': 5}),
    tee= True,
    get_labeled_model_args=None,
    _Cholesky_option=True,
    _only_compute_fim_lower=True,
    )

    doe_obj.run_doe()

    if run == 1: ## Do this only on run 1
        ## Header names
        HEADERS = [
            "run",
            "solve_time",
            "build_time",
            "init_time",
            "wall_time",
            "objective_value"
        ] + list(doe_obj.results["Experiment Design Names"])
        
        ws.append(HEADERS)

    row = [
        run,
        doe_obj.results["Solve Time"],
        doe_obj.results["Build Time"],
        doe_obj.results["Initialization Time"],
        doe_obj.results["Wall-clock Time"],
        pyo.value(doe_obj.model.objective)
    ] + list(doe_obj.results["Experiment Design"])

    ws.append(row)
    
wb.save(XLSX_PATH)



# Print out a results summary
print("Optimal experiment values: ")
print(
    "\tInitial state: {:.2f}".format(
        doe_obj.results["Experiment Design"][0]
    )
)
print(
    ("\t Control input values: [" + "{:.2f}, " * 8 + "{:.2f}]").format(
        *doe_obj.results["Experiment Design"][1:]
    )
)
print("FIM at optimal design:\n {}".format(np.array(doe_obj.results["FIM"])))
print(
    "Objective value at optimal design: {:.2f}".format(
        pyo.value(doe_obj.model.objective)
    )
)

print(doe_obj.results["Experiment Design Names"])

#print(sorted(doe_obj.results.keys()))

print("Solve time (s):", doe_obj.results["Solve Time"])
print("Build time (s):", doe_obj.results["Build Time"])
print("Initialization time (s):", doe_obj.results["Initialization Time"])
print("Total wall time (s):", doe_obj.results["Wall-clock Time"])

###################
# End optimal DoE




Run 1/1000
Ipopt 3.13.2: linear_solver=ma57


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
  